In [ ]:
from sklearn.svm import SVR
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_log_error, mean_squared_error
from sklearn.model_selection import train_test_split, GridSearchCV
import pandas as pd
import numpy as np

In [ ]:
df_train = pd.read_csv('../input/house-prices-advanced-regression-techniques/train.csv')
y_train = df_train.loc[:, 'SalePrice'].to_numpy()
df_train = df_train.drop(columns='SalePrice')
df_test = pd.read_csv('../input/house-prices-advanced-regression-techniques/test.csv')
df = pd.concat([df_train, df_test], axis=0)
df = df.drop(columns='Id')
df

In [ ]:
bad_cols = df.columns[df.iloc[:, :].isnull().sum() > 500]
df = df.drop(bad_cols, axis=1)
df

In [ ]:
obj_cols = df.loc[:, df.dtypes == object]
s1 = df.mean(numeric_only=True)
s2 = obj_cols.mode().loc[0]
s = pd.concat([s1, s2], axis=0)
df = df.fillna(value=s)
df

In [ ]:
dummies = pd.get_dummies(obj_cols)
df = pd.concat([df, dummies], axis=1)
df = df.drop(obj_cols, axis=1)
df

In [ ]:
x = df.to_numpy()
scaler = StandardScaler()
scaler.fit(x)
x = scaler.transform(x)
x_train = x[0: y_train.shape[0]]
x_test = x[y_train.shape[0]: ]

In [ ]:
'''svr = SVR()
param_grid = {'kernel': ['linear', 'poly', 'rbf'], 'C': [0.1, 0.5, 1, 10, 50, 100]}
svr = GridSearchCV(svr, param_grid, cv=5, scoring='neg_mean_squared_log_error')
svr.fit(x, y)
print(svr.best_score_)
print(svr.best_params_)'''

In [ ]:
svr = SVR(kernel='linear', C=50)
svr.fit(x_train, y_train)
svr_pred = svr.predict(x_train)
mean_squared_log_error(y_train, svr_pred)

In [ ]:
'''g_boost = GradientBoostingRegressor()
param_grid = {'learning_rate': [0.001, 0.01, 0.1], 'n_estimators': [50, 100, 200, 300]}
g_boost = GridSearchCV(g_boost, param_grid, cv=5, scoring='neg_mean_squared_log_error')
g_boost.fit(x, y)
print(g_boost.best_score_)
print(g_boost.best_params_)'''

In [ ]:
g_boost = GradientBoostingRegressor(learning_rate=0.1, n_estimators=300)
g_boost.fit(x_train, y_train)
g_boost_pred = g_boost.predict(x_train)
mean_squared_log_error(y_train, g_boost_pred)

In [ ]:
sample_sub = pd.read_csv('../input/house-prices-advanced-regression-techniques/sample_submission.csv')
pred1 = g_boost.predict(x_test)
pred2 = svr.predict(x_test)
sample_sub.loc[:, 'SalePrice'] = (pred1 + pred2) / 2
sample_sub.to_csv('submission.csv',index=False)